# 14 - Primary ECG and secondary CXR reporting analysis

Primary automated endpoint: diagnostic micro-F1 agreement with imperfect references. Report per-label
metrics, report availability, denominators, and reference-discordant positives (not adjudicated hallucinations).
Within-model linked vs same-sensor and linked vs random contrasts come first. Cross-pipeline comparisons
are secondary: inputs, prompts and evidence budgets differ. Paired quality uses common successful,
reference-evaluable examinations and patient-cluster bootstrap intervals (1000 replicates by default).
PULSE schema/attribution metrics are not applicable. Uncertain/missing CXR labels are masked, not negatives.
An exploratory exact-reference-agreement yield includes failures in its denominator without calling
technical failures incorrect diagnoses. No clinical safety or sustainability gate is inferred.
Unattempted cells block final analysis; terminal failures do not. Set `E4_ALLOW_INCOMPLETE=1` only for
a clearly preliminary snapshot. New outputs/code/settings after freezing require a new E4_ANALYSIS_ID.
Broken-linkage, ECG-only retrieval, controlled timing/energy and unexecuted robustness studies are
reported as unavailable in `manuscript_readiness.csv`, not invented from existing conditions.

In [ ]:
from pathlib import Path
import os, sys
CODE_ROOT = Path(os.environ.get('EXPERIMENT4_CODE_ROOT', str(Path.cwd())))
if not (CODE_ROOT / 'e4_common.py').is_file() and 'EXPERIMENT4_CODE_ROOT' not in os.environ:
    CODE_ROOT = Path.cwd()
if not CODE_ROOT.is_absolute() or not (CODE_ROOT / 'e4_common.py').is_file():
    raise FileNotFoundError('Set EXPERIMENT4_CODE_ROOT to the uploaded Experiment 4 source directory')
if str(CODE_ROOT) not in sys.path: sys.path.insert(0, str(CODE_ROOT))
from e4_common import ARTIFACT_ROOT, EXPERIMENT4_ROOT, SYMILE_ROOT
print({'code_root': str(CODE_ROOT), 'artifact_root': str(ARTIFACT_ROOT),
       'experiment4_root': str(EXPERIMENT4_ROOT), 'symile_root': str(SYMILE_ROOT)})

In [ ]:
# Same selection file and analysis ID in notebooks 13E-16; independent of generation run env.
os.environ['E4_ANALYSIS_CONFIG'] = str(CODE_ROOT / 'analysis_model_runs.json')
os.environ.setdefault('E4_ANALYSIS_ID', 'jbhi_five_models_v1')
from e4_jbhi_analysis import configuration, analysis_path
config = configuration()
print('Model-to-run mapping:', config['model_runs'])
print('Analysis output:', analysis_path(EXPERIMENT4_ROOT, config))

In [ ]:
from e4_evaluation import extractor_self_test
from e4_jbhi_analysis import evaluate
print(extractor_self_test())
summary = evaluate(allow_incomplete=os.environ.get('E4_ALLOW_INCOMPLETE', '0').lower() in {'1','true','yes'})
display(summary[summary.direction.eq('ecg')])
display(summary[summary.direction.eq('cxr')])

In [ ]:
import pandas as pd
root = analysis_path(EXPERIMENT4_ROOT, config)
paired = pd.read_csv(root / 'paired_comparisons.csv')
display(paired[paired.comparison_type.eq('within_model') & paired.direction.eq('ecg')])
display(paired[paired.comparison_type.eq('cross_pipeline')])
display(pd.read_csv(root / 'manuscript_readiness.csv'))